# Guía práctica 3: análisis de calidad del agua

**Estadística Descriptiva · Ingeniería Ambiental**

Este cuaderno reproduce en Python el análisis del script R usando `Base_datos_Guia_3_calidad_agua.xlsx`. Está preparado para ejecutarse de principio a fin en Google Colab.

**Problema:** describir la concentración de nitratos en puntos de monitoreo de una microcuenca de Loja y explorar su variación con otras características ambientales. Los datos son sintéticos; el umbral de 7,5 mg/L es didáctico y no representa un límite normativo.

## 1. Preparación y carga del libro

En Colab, la celda solicita cargar el archivo Excel desde el equipo. Si el libro ya está en el directorio de trabajo (por ejemplo, al ejecutar localmente), lo utiliza directamente.

In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='deep')

nombre_excel = 'Base_datos_Guia_3_calidad_agua.xlsx'
rutas_candidatas = [
    Path(nombre_excel),
    Path('Estadistica_2026/01_unidad_1/01_guias/Unidad_1') / nombre_excel,
]
ruta_excel = next((ruta for ruta in rutas_candidatas if ruta.exists()), None)
if ruta_excel is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            f'No se encontró {nombre_excel}. Coloque el Excel junto al cuaderno o ejecútelo en Colab y cárguelo.'
        ) from error
    archivos = files.upload()
    if nombre_excel not in archivos:
        raise FileNotFoundError(f'Cargue el archivo con este nombre exacto: {nombre_excel}')
    ruta_excel = Path(nombre_excel)

print(f'Libro seleccionado: {ruta_excel.resolve()}')

In [ ]:
datos = pd.read_excel(ruta_excel, sheet_name='Datos')
columnas_requeridas = {
    'id_punto', 'zona', 'subcuenca', 'uso_suelo', 'fecha_muestreo',
    'altitud_m', 'lluvia_24h_mm', 'nitratos_mg_l', 'ph', 'turbidez_ntu',
    'conductividad_us_cm', 'temperatura_c', 'alerta_nitratos', 'seleccion_muestra'
}
faltantes = columnas_requeridas.difference(datos.columns)
if faltantes:
    raise ValueError(f'Faltan columnas requeridas en la hoja Datos: {sorted(faltantes)}')

datos['fecha_muestreo'] = pd.to_datetime(datos['fecha_muestreo'], errors='coerce')
muestra = datos.loc[datos['seleccion_muestra'].eq('Sí')].copy()
N = len(datos)
n = len(muestra)
if datos['id_punto'].nunique() != N:
    raise ValueError('El marco contiene identificadores de punto duplicados.')
if N != 120 or n != 35:
    raise ValueError(f'Se esperaban 120 puntos y 35 seleccionados; se encontraron N={N}, n={n}.')
if muestra['nitratos_mg_l'].isna().any():
    raise ValueError('Hay nitratos faltantes en la muestra; revise los registros antes de estimar.')

print(f'Marco: N={N} puntos únicos | Muestra aleatoria simple: n={n}')
print('Selección por zona:')
display(pd.crosstab(datos['zona'], datos['seleccion_muestra']))
display(datos.head())

## 2. Problema, diseño y unidades

La población objetivo son los puntos elegibles de monitoreo de la microcuenca durante la campaña definida. El marco contiene 120 puntos; se dispone de una muestra aleatoria simple sin reemplazo de 35. El estudio es observacional y transversal: no asigna tratamientos ni manipula la calidad del agua.

Cada fila representa un punto y es la unidad muestral y de análisis. La muestra de agua y sus lecturas de campo o laboratorio son unidades de observación asociadas a ese punto. Las mediciones repetidas no aumentan el número de puntos independientes.

`alerta_nitratos` indica si los nitratos son iguales o superiores a 7,5 mg/L. Es un umbral creado para el ejercicio, no una norma ambiental.

## 3. Preparación y control de calidad

Los datos faltantes y los valores atípicos se identifican para revisión; no se eliminan ni imputan automáticamente. Un valor estadísticamente extremo puede ser real y debe contrastarse con la ficha, el equipo o el laboratorio.

In [ ]:
faltantes_por_variable = datos.isna().sum().sort_values(ascending=False)
duplicados = datos.loc[datos['id_punto'].duplicated(keep=False), 'id_punto']
ph_fuera_rango = datos.loc[datos['ph'].notna() & ~datos['ph'].between(0, 14), ['id_punto', 'ph']]
nitratos_no_positivos = datos.loc[datos['nitratos_mg_l'].notna() & datos['nitratos_mg_l'].le(0), ['id_punto', 'nitratos_mg_l']]

q1, q3 = datos['turbidez_ntu'].quantile([0.25, 0.75])
riq = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * riq, q3 + 1.5 * riq
atipicos_turbidez = datos.loc[~datos['turbidez_ntu'].between(limite_inferior, limite_superior), ['id_punto', 'turbidez_ntu']]

print('Faltantes por variable:')
display(faltantes_por_variable.to_frame('faltantes'))
print(f'IDs duplicados: {len(duplicados)}')
print(f'pH fuera de [0, 14]: {len(ph_fuera_rango)}')
display(ph_fuera_rango)
print(f'Nitratos no positivos: {len(nitratos_no_positivos)}')
display(nitratos_no_positivos)
print(f'Posibles atípicos de turbidez por regla 1,5 × RIQ: {len(atipicos_turbidez)} (revisar; no borrar automáticamente)')
display(atipicos_turbidez)
print('El faltante de conductividad se debe verificar con la ficha o el laboratorio; no se imputará sin justificación.')

## 4. Tamaño de muestra y precisión

Se calcula el tamaño para estimar una media con población finita y el tamaño para una proporción. Para la proporción se comparan una estimación previa de $p=0,30$ y el escenario conservador $p=0,50$, que maximiza $p(1-p)$. Los tamaños se redondean hacia arriba.

In [ ]:
confianza = 0.95
Z = stats.norm.ppf(1 - (1 - confianza) / 2)
sigma_piloto = 3.5
error_media = 1.0
n_media_infinita = Z**2 * sigma_piloto**2 / error_media**2
n_media_finita = N * Z**2 * sigma_piloto**2 / ((N - 1) * error_media**2 + Z**2 * sigma_piloto**2)

def tamano_proporcion(p, error, poblacion=N, z=Z):
    q = 1 - p
    return poblacion * z**2 * p * q / (error**2 * (poblacion - 1) + z**2 * p * q)

print(f'Z para 95% de confianza: {Z:.3f}')
print(f'Tamaño para la media, población grande: {n_media_infinita:.2f} → {int(np.ceil(n_media_infinita))}')
print(f'Tamaño para la media, población finita: {n_media_finita:.2f} → {int(np.ceil(n_media_finita))}; se trabaja con n=35.')

escenarios = pd.DataFrame([
    {'p': p, 'q': 1-p, 'error': e, 'n_sin_redondear': tamano_proporcion(p, e)}
    for p, e in itertools.product([0.30, 0.50], [0.05, 0.10, 0.15])
])
escenarios['n_requerido'] = np.ceil(escenarios['n_sin_redondear']).astype(int)
display(escenarios.round(3))

valores_z = pd.DataFrame({'confianza': [0.90, 0.95, 0.97, 0.98, 0.99]})
valores_z['Z'] = stats.norm.ppf((1 + valores_z['confianza']) / 2)
display(valores_z.assign(confianza_pct=100*valores_z['confianza'])[['confianza_pct', 'Z']].round(3))

## 5. Estimación e intervalos de confianza

Se aplica la corrección por población finita correspondiente al muestreo aleatorio simple sin reemplazo. Para la media se usa un intervalo $t$; para la proporción, un intervalo normal aproximado. Los parámetros calculados con los 120 registros solo se conocen porque el conjunto es sintético y está completo.

In [ ]:
fpc = np.sqrt((N - n) / (N - 1))
z_critico = stats.norm.ppf(0.975)

media_muestral = muestra['nitratos_mg_l'].mean()
sd_muestral = muestra['nitratos_mg_l'].std(ddof=1)
se_media = sd_muestral / np.sqrt(n) * fpc
t_critico = stats.t.ppf(0.975, df=n-1)
ic_media = (media_muestral - t_critico*se_media, media_muestral + t_critico*se_media)
media_marco = datos['nitratos_mg_l'].mean()

proporcion_muestral = muestra['alerta_nitratos'].eq('Sí').mean()
proporcion_marco = datos['alerta_nitratos'].eq('Sí').mean()
se_proporcion = np.sqrt(proporcion_muestral*(1-proporcion_muestral)/n) * fpc
ic_proporcion = (max(0, proporcion_muestral-z_critico*se_proporcion),
                  min(1, proporcion_muestral+z_critico*se_proporcion))

estimaciones = pd.DataFrame([
    {'cantidad': 'Media de nitratos (mg/L)', 'estimación muestral': media_muestral, 'IC 95% inferior': ic_media[0], 'IC 95% superior': ic_media[1], 'valor del marco sintético': media_marco},
    {'cantidad': 'Proporción con alerta', 'estimación muestral': proporcion_muestral, 'IC 95% inferior': ic_proporcion[0], 'IC 95% superior': ic_proporcion[1], 'valor del marco sintético': proporcion_marco}
])
display(estimaciones.round(3))
print(f'FPC = {fpc:.3f}; media muestral = {media_muestral:.2f} mg/L; proporción muestral = {proporcion_muestral:.3f}.')

## 6. Descripción y comparación entre grupos

Las tablas resumen la distribución de las variables cuantitativas y describen nitratos por zona y uso de suelo. Las diferencias son descriptivas; los grupos no fueron asignados al azar como tratamientos.

In [ ]:
variables_numericas = ['nitratos_mg_l', 'ph', 'turbidez_ntu', 'conductividad_us_cm', 'temperatura_c', 'lluvia_24h_mm', 'altitud_m']
resumen_numerico = datos[variables_numericas].describe().T
resumen_numerico = resumen_numerico.rename(columns={'count':'válidos', 'mean':'media', 'std':'desviación', 'min':'mínimo', '25%':'Q1', '50%':'mediana', '75%':'Q3', 'max':'máximo'})
resumen_numerico['faltantes'] = datos[variables_numericas].isna().sum()
display(resumen_numerico.round(3))

def resumen_por_grupo(columna):
    return (muestra.groupby(columna, observed=True)['nitratos_mg_l']
            .agg(n='count', media='mean', mediana='median', desviación='std', mínimo='min', máximo='max')
            .round(3))

print('Nitratos por zona en la muestra:')
display(resumen_por_grupo('zona'))
print('Nitratos por uso de suelo en la muestra:')
display(resumen_por_grupo('uso_suelo'))

frecuencias = pd.crosstab(muestra['uso_suelo'], muestra['alerta_nitratos'].eq('Sí'))
frecuencias.columns = ['sin_alerta', 'con_alerta'] if len(frecuencias.columns) == 2 else frecuencias.columns
display(frecuencias)

## 7. Contrastes y asociaciones exploratorias

Welch compara medias entre zonas sin asumir varianzas iguales; Kruskal–Wallis compara rangos. Las correlaciones de Spearman exploran asociaciones monotónicas. Debido al tamaño muestral y al carácter observacional, los valores *p* son exploratorios: no prueban causalidad ni deben interpretarse aislados del diseño y la calidad de medición.

In [ ]:
def anova_welch(grupos):
    grupos = [np.asarray(g.dropna(), dtype=float) for g in grupos]
    k = len(grupos)
    n_i = np.array([len(g) for g in grupos], dtype=float)
    medias = np.array([g.mean() for g in grupos])
    varianzas = np.array([g.var(ddof=1) for g in grupos])
    pesos = n_i / varianzas
    media_pond = np.sum(pesos * medias) / np.sum(pesos)
    termino = np.sum((1 - pesos / np.sum(pesos)) ** 2 / (n_i - 1))
    estadistico = (np.sum(pesos * (medias - media_pond) ** 2) / (k - 1)) / (
        1 + 2 * (k - 2) / (k**2 - 1) * termino
    )
    gl_num = k - 1
    gl_den = (k**2 - 1) / (3 * termino)
    return {
        'F': estadistico,
        'gl_num': gl_num,
        'gl_den': gl_den,
        'p_valor': stats.f.sf(estadistico, gl_num, gl_den),
    }

grupos_zona = [
    grupo['nitratos_mg_l']
    for _, grupo in muestra.groupby('zona', observed=True)
]
print('ANOVA de Welch por zona:')
display(pd.DataFrame([anova_welch(grupos_zona)]))
print('Kruskal–Wallis por zona:')
kruskal = stats.kruskal(*grupos_zona)
print(f'H={kruskal.statistic:.3f}, gl={len(grupos_zona)-1}, p={kruskal.pvalue:.4f}')

variables_asociacion = [
    'lluvia_24h_mm', 'altitud_m', 'turbidez_ntu',
    'temperatura_c', 'conductividad_us_cm'
]
filas_correlacion = []
for variable in variables_asociacion:
    pares = muestra[['nitratos_mg_l', variable]].dropna()
    resultado = stats.spearmanr(pares['nitratos_mg_l'], pares[variable])
    filas_correlacion.append({
        'variable': variable,
        'n': len(pares),
        'rho_spearman': resultado.statistic,
        'p_valor': resultado.pvalue,
    })
correlaciones = pd.DataFrame(filas_correlacion).set_index('variable')
display(correlaciones.round(4))

## 8. Visualización exploratoria

Los cuatro gráficos complementan las tablas: distribución de nitratos, diferencias descriptivas por zona, relación con la lluvia antecedente y proporción de alertas por uso de suelo. La línea de 7,5 mg/L es solo una referencia didáctica, no un límite normativo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=datos, x='nitratos_mg_l', bins=12, color='#2E6F62', ax=axes[0, 0])
axes[0, 0].axvline(7.5, color='#A52828', linestyle='--', label='Umbral didáctico: 7,5 mg/L')
axes[0, 0].set(title='Distribución de nitratos en el marco', xlabel='Nitratos (mg/L)', ylabel='Puntos')
axes[0, 0].legend()

sns.boxplot(data=muestra, x='zona', y='nitratos_mg_l', color='#BFD8D0', ax=axes[0, 1])
sns.stripplot(data=muestra, x='zona', y='nitratos_mg_l', color='#163A5F', alpha=0.65, jitter=0.15, ax=axes[0, 1])
axes[0, 1].set(title='Nitratos por zona: muestra', xlabel='Zona', ylabel='Nitratos (mg/L)')

sns.regplot(data=muestra, x='lluvia_24h_mm', y='nitratos_mg_l', lowess=True,
            scatter_kws={'alpha': 0.75}, line_kws={'color': '#C66E26'}, ax=axes[1, 0])
axes[1, 0].set(title='Lluvia antecedente y nitratos', xlabel='Lluvia en 24 h (mm)', ylabel='Nitratos (mg/L)')

alertas_uso = (muestra.assign(con_alerta=muestra['alerta_nitratos'].eq('Sí'))
               .groupby('uso_suelo', observed=True)['con_alerta'].mean().reset_index())
sns.barplot(data=alertas_uso, x='uso_suelo', y='con_alerta', color='#2E6F62', ax=axes[1, 1])
axes[1, 1].set(ylim=(0, 1), title='Proporción con alerta por uso de suelo',
               xlabel='Uso de suelo', ylabel='Proporción de la muestra')

fig.suptitle('Exploración de la calidad del agua', fontsize=15)
fig.tight_layout()
plt.show()

## 9. Síntesis para el informe

Redacte una conclusión con la media muestral y su intervalo, la proporción de puntos sobre el umbral didáctico y su intervalo, y al menos una limitación del diseño. Describa diferencias y asociaciones como exploratorias; no atribuya causalidad. El marco de 120 registros es sintético y se utiliza únicamente para aprender a contrastar estimadores con valores conocidos.